In [ ]:
import sys
sys.path.append('/Users/wayne/Desktop/Cluster/ClusterHome/gitrepo/2024/OncoTRAIL/src')
from constants import (target_dict_mapping,
                        cancer_novelty_map, 
                        epic_map_cancer_to_group, 
                        epr_map_cancer_to_group,
                        CANCER_CODE_MAP,
                        epr_CHEMO_DRUG_MAP,
                        epic_CHEMO_DRUG_MAP)
import pandas as pd
import numpy as np

In [ ]:
def mrn_inference_aero_from_anchored_notes(path_to_anchored_notes):
    df_anchored_notes = pd.read_csv(path_to_anchored_notes)
    df_anchored_notes['cancer_novelty'] = df_anchored_notes['primary_site_desc'].map(cancer_novelty_map)

    mrn_inference_aero = df_anchored_notes[df_anchored_notes['cancer_novelty'] == 0]['mrn'].unique().tolist()

    return mrn_inference_aero

def process_id_cancer_sites(df):
    df = df.copy()
        
    # Extract cancer columns
    cancer_cols = [col for col in df.columns if col.startswith("cancer_site_C")]

    # Melt into long format
    df_long = df.melt(
        id_vars=["treatment_date", 'mrn'],
        value_vars=cancer_cols,
        var_name="Cancer Code",
        value_name="Has Cancer"
    )

    df_long = df_long[df_long["Has Cancer"] == 1].copy()

    # Extract cancer code and map to type
    df_long["Cancer Code"] = df_long["Cancer Code"].str.replace("cancer_site_", "")
    df_long["Cancer Type"] = df_long["Cancer Code"].map(CANCER_CODE_MAP).fillna(df_long["Cancer Code"])
    df_long['Cancer Type'] = df_long['Cancer Type'].replace(epr_map_cancer_to_group)
    # if Has Cancer is 0, set Cancer Type to "Other/Ill-defined"
    df_long.loc[df_long["Has Cancer"] == 0, "Cancer Type"] = "Other/Ill-defined"

    return df_long

def process_od_cancer_sites(df):
    df['Cancer Type'] = df['primary_site_desc'].astype(str).apply(epic_map_cancer_to_group)
    
    return df[['mrn', 'treatment_date', 'Cancer Type']]

def process_target_columns(df):
    target_cols = [col for col in df.columns if col.startswith("target")]
    id_vars = [col for col in df.columns if col not in target_cols]
    # exclude any target column that contains (1pt_change|grade3plus|ecog)
    target_cols = [col for col in target_cols if not any(x in col for x in ["1pt_change", "grade3plus", "ecog"])]

    # keep all other columns and target_cols
    df = df[id_vars + target_cols].copy()
    # rename target columns such that '_' is replaced by '-'
    df.rename(columns=lambda x: x.replace("_", "-") if x.startswith("target") else x, inplace=True)
    # map target columns to more interpretable names
    df.rename(columns=target_dict_mapping, inplace=True)
    return df

def get_mrn_treatment_df(df):
    df = df.copy()
    df['treatment_date'] = (
        pd.to_datetime(df['treatment_date'], utc=True)
        .dt.normalize()
    )
    df['mrn'] = df['mrn'].astype(str)

    return (
        df[['mrn', 'treatment_date']]
        .drop_duplicates()
        .reset_index(drop=True)
    )

def process_drug_name(drug_df, mrn_treat_df, data_source):
    drug_df = drug_df.copy()

    if data_source == "epr":
        mrn_col = 'Hosp_Chart'
        treatment_col = 'Trt_Date'
        drug_col = 'Drug_name'
        drug_mapping = epr_CHEMO_DRUG_MAP
    elif data_source == "epic":
        mrn_col = 'mrn'
        treatment_col = 'treatment_date'
        drug_col = 'drug_name_normalized'
        drug_mapping = epic_CHEMO_DRUG_MAP
    else:
        raise ValueError(f"Unknown data_source: {data_source}")

    drug_list = set(drug_mapping.keys())

    # ---- standardize join keys ----
    drug_df[mrn_col] = drug_df[mrn_col].astype(str)
    drug_df[treatment_col] = (
        pd.to_datetime(drug_df[treatment_col], utc=True)
        .dt.normalize()
    )

    # ---- rename for clean join ----
    drug_df = drug_df.rename(
        columns={mrn_col: 'mrn', treatment_col: 'treatment_date'}
    )

    # ---- inner join on (mrn, treatment_date) ----
    drug_df_filtered = drug_df.merge(
        mrn_treat_df,
        on=['mrn', 'treatment_date'],
        how='inner'
    )

    # ---- keep only chemotherapy drugs ----
    drug_df_filtered = drug_df_filtered[
        drug_df_filtered[drug_col].isin(drug_list)
    ].copy()

    drug_df_filtered['Standardized Drug'] = (
        drug_df_filtered[drug_col].map(drug_mapping)
    )

    # make mrn int again
    drug_df_filtered['mrn'] = drug_df_filtered['mrn'].astype(int)

    return drug_df_filtered[['mrn', 'treatment_date', 'Standardized Drug']]

In [ ]:
def build_characteristics_table(
    df_id,
    df_od,
    id_cancer,
    od_cancer,
    id_chemo,
    od_chemo,
    target_ae_cols,
    split_date,
    mrn_inference_aero   # NEW
):
    
    split_date = pd.to_datetime(split_date, utc=True)

    # -------------------------
    # Helper functions
    # -------------------------
    def split_id_df(df):
        df = df.copy()
        df["treatment_date"] = pd.to_datetime(df["treatment_date"], utc=True)
        train = df[df["treatment_date"] <= split_date]
        temporal = df[df["treatment_date"] > split_date]
        return train, temporal

    def format_count(count, denom):
        pct = 100 * count / denom if denom > 0 else 0
        return f"{count:,} ({pct:.2f})"

    def format_median_iqr(series):
        med = np.median(series)
        q1 = np.percentile(series, 25)
        q3 = np.percentile(series, 75)
        return f"{med:.2f} ({q1:.2f}-{q3:.2f})"

    def frequency_table(df, col, total_patients):
        counts = df[col].value_counts().sort_index()
        out = {}
        for k, v in counts.items():
            out[k] = format_count(v, total_patients)
        return out, counts

    # -------------------------
    # Split data
    # -------------------------
    id_train, id_temporal = split_id_df(df_id)
    cancer_train, cancer_temporal = split_id_df(id_cancer)
    chemo_train, chemo_temporal = split_id_df(id_chemo)

    # -------------------------
    # Split OOD by cancer group
    # -------------------------
    df_od_aero = df_od[df_od["mrn"].isin(mrn_inference_aero)]
    df_od_other = df_od[~df_od["mrn"].isin(mrn_inference_aero)]

    # -------------------------
    # Column containers
    # -------------------------
    # columns = {
    #     "Training": {},
    #     "Temporal OOD": {},
    #     "Multi-factor OOD": {}
    # }

    columns = {
        "Training": {},
        "Temporal OOD": {},
        "Multi-factor OOD": {},
        "OOD Aero": {},          # NEW
        "OOD Other": {}          # NEW
    }

    # =========================
    # Demographics
    # =========================
    # splits = {
    #     "Training": id_train,
    #     "Temporal OOD": id_temporal,
    #     "Multi-factor OOD": df_od
    # }

    splits = {
        "Training": id_train,
        "Temporal OOD": id_temporal,
        "Multi-factor OOD": df_od,
        "OOD Aero": df_od_aero,        # NEW
        "OOD Other": df_od_other       # NEW
    }

    for name, df in splits.items():
        n = df.shape[0]

        columns[name]["Age, median (IQR)"] = format_median_iqr(df["age"])
        female_n = df["female"].sum()
        columns[name]["Female"] = format_count(female_n, n)

    # =========================
    # Adverse events
    # =========================
    for ae in target_ae_cols:
        for name, df in splits.items():
            valid = df[df[ae] != -1]
            denom = valid.shape[0]
            count = (valid[ae] == 1).sum()
            columns[name][ae] = format_count(count, denom)

    # =========================
    # Cancer type
    # =========================
    # cancer_splits = {
    #     "Training": cancer_train,
    #     "Temporal OOD": cancer_temporal,
    #     "Multi-factor OOD": od_cancer
    # }

    od_cancer_aero = od_cancer[od_cancer["mrn"].isin(mrn_inference_aero)]
    od_cancer_other = od_cancer[~od_cancer["mrn"].isin(mrn_inference_aero)]

    cancer_splits = {
        "Training": cancer_train,
        "Temporal OOD": cancer_temporal,
        "Multi-factor OOD": od_cancer,
        "OOD Aero": od_cancer_aero,
        "OOD Other": od_cancer_other
    }

    cancer_counts = {}

    for name, df in cancer_splits.items():
        total = df['mrn'].nunique()
        freq, raw = frequency_table(df, "Cancer Type", total)
        columns[name].update({f"Cancer: {k}": v for k, v in freq.items()})
        if name == "Training":
            cancer_counts = raw

    # =========================
    # Chemotherapy drugs
    # =========================
    # chemo_splits = {
    #     "Training": chemo_train,
    #     "Temporal OOD": chemo_temporal,
    #     "Multi-factor OOD": od_chemo
    # }

    od_chemo_aero = od_chemo[od_chemo["mrn"].isin(mrn_inference_aero)]
    od_chemo_other = od_chemo[~od_chemo["mrn"].isin(mrn_inference_aero)]

    chemo_splits = {
        "Training": chemo_train,
        "Temporal OOD": chemo_temporal,
        "Multi-factor OOD": od_chemo,
        "OOD Aero": od_chemo_aero,
        "OOD Other": od_chemo_other
    }

    drug_counts = {}

    for name, df in chemo_splits.items():
        # unique patients per drug
        df_unique = df.drop_duplicates(subset=["mrn", "Standardized Drug"])
        total_patients = df_unique["mrn"].nunique()

        counts = (
            df_unique
            .groupby("Standardized Drug")["mrn"]
            .nunique()
            .sort_values(ascending=False)
        )

        for drug, cnt in counts.items():
            columns[name][f"Chemo: {drug}"] = format_count(cnt, total_patients)

        if name == "Training":
            drug_counts = counts

    # =========================
    # Build final table
    # =========================
    table = pd.DataFrame(columns).fillna("0 (0)")

    # Ordering: demographics → AEs → cancer → chemo
    ordering = []

    ordering += ["Age, median (IQR)", "Female"]
    ordering += target_ae_cols

    # --- collect all cancer types across splits ---
    all_cancers = set()
    for df in cancer_splits.values():
        all_cancers.update(df["Cancer Type"].dropna().unique())

    # --- training counts ---
    train_cancer_counts = (
        cancer_train["Cancer Type"]
        .value_counts()
        .to_dict()
    )

    # --- ordering: descending training freq, then name ---
    ordered_cancers = sorted(
        all_cancers,
        key=lambda x: (-train_cancer_counts.get(x, 0), x)
    )

    ordering += [f"Cancer: {k}" for k in ordered_cancers]

    # --- collect all drugs across splits ---
    all_drugs = set()
    for df in chemo_splits.values():
        all_drugs.update(df["Standardized Drug"].dropna().unique())

    # --- training drug counts (patients per drug) ---
    train_drug_counts = (
        chemo_train
        .drop_duplicates(subset=["mrn", "Standardized Drug"])
        .groupby("Standardized Drug")["mrn"]
        .nunique()
        .to_dict()
    )

    # --- ordering: descending training freq, then name ---
    ordered_drugs = sorted(
        all_drugs,
        key=lambda x: (-train_drug_counts.get(x, 0), x)
    )

    ordering += [f"Chemo: {k}" for k in ordered_drugs]

    table = table.loc[ordering]

    # =========================
    # Create second table with limited chemo drugs
    # =========================

    # --- training top 5 ---
    train_top5 = (
        chemo_train
        .drop_duplicates(subset=["mrn", "Standardized Drug"])
        .groupby("Standardized Drug")["mrn"]
        .nunique()
        .sort_values(ascending=False)
        .head(5)
        .index
    )

    print(train_top5)

    # --- OOD top 5 ---
    ood_top5 = (
        od_chemo
        .drop_duplicates(subset=["mrn", "Standardized Drug"])
        .groupby("Standardized Drug")["mrn"]
        .nunique()
        .sort_values(ascending=False)
        .head(5)
        .index
    )

    print(ood_top5)

    # ordering: training first then OOD
    selected_drugs = list(train_top5) + [d for d in ood_top5 if d not in train_top5]

    drug_rows = [f"Chemo: {d}" for d in selected_drugs]

    # keep all non-chemo rows
    non_chemo_rows = [r for r in table.index if not r.startswith("Chemo:")]

    table_top_drugs = table.loc[non_chemo_rows + drug_rows]

    return table, table_top_drugs

In [6]:
# open the anchored data
# df_id = pd.read_csv(
#     '/Users/wayne/Desktop/Cluster/WorkDirHome/gitrepo/2024/OncoTRAIL/paper/pmh_method/data/train_test/note_anchored/note_anchored_firstTreatmentOnly-medOnc-ConsultLetterClinic_deid.csv'
# )
df_id = pd.read_csv(
    '/Users/wayneisaacuy/Desktop/UHN/WorkDirHome/gitrepo/2024/OncoTRAIL/paper/pmh_method/data/train_test/note_anchored/note_anchored_firstTreatmentOnly-medOnc-ConsultLetterClinic_deid.csv'
)
df_id['female'] = df_id['female'].astype(int)
df_id = process_target_columns(df_id)
id_mrn_treat_df = get_mrn_treatment_df(df_id)
# df_od = pd.read_csv(
#     '/Users/wayne/Desktop/Cluster/WorkDirHome/gitrepo/2024/OncoTRAIL/paper/pmh_method/data/inference/note_anchored/note_anchored_firstTreatmentOnly-medOnc-ConsultLetterClinic_deid.csv'
# )
df_od = pd.read_csv(
    '/Users/wayneisaacuy/Desktop/UHN/WorkDirHome/gitrepo/2024/OncoTRAIL/paper/pmh_method/data/inference/note_anchored/note_anchored_firstTreatmentOnly-medOnc-ConsultLetterClinic_deid.csv'
)
df_od = process_target_columns(df_od)
od_mrn_treat_df = get_mrn_treatment_df(df_od)
# mrn_inference_aero = mrn_inference_aero_from_anchored_notes('/Users/wayne/Desktop/Cluster/WorkDirHome/gitrepo/2024/OncoTRAIL/paper/pmh_method/data/inference/note_anchored/note_anchored_firstTreatmentOnly-medOnc-ConsultLetterClinic_deid.csv')
mrn_inference_aero = mrn_inference_aero_from_anchored_notes('/Users/wayneisaacuy/Desktop/UHN/WorkDirHome/gitrepo/2024/OncoTRAIL/paper/pmh_method/data/inference/note_anchored/note_anchored_firstTreatmentOnly-medOnc-ConsultLetterClinic_deid.csv')

In [7]:
# chemo data
# id_chemo = pd.read_parquet(
#     "/Users/wayne/Desktop/Cluster/H4Hhome/2BLAST/data/final/data_2023-02-21/raw/opis.parquet.gzip"
# )
id_chemo = pd.read_parquet(
    "/Users/wayneisaacuy/Desktop/UHN/H4Hhome/2BLAST/data/final/data_2023-02-21/raw/opis.parquet.gzip"
)
id_chemo = process_drug_name(
    id_chemo, id_mrn_treat_df, data_source="epr"
)
# od_chemo = pd.read_parquet(
#     "/Users/wayne/Desktop/Cluster/H4Hhome/2BLAST/data/final/data_2025-03-29/interim/chemo.parquet"
# )
od_chemo = pd.read_parquet(
    "/Users/wayneisaacuy/Desktop/UHN/H4Hhome/2BLAST/data/final/data_2025-03-29/interim/chemo.parquet"
)
od_chemo = process_drug_name(
    od_chemo, od_mrn_treat_df, data_source="epic"
)

# cancer site
id_cancer = process_id_cancer_sites(df_id)
od_cancer = process_od_cancer_sites(df_od)

In [36]:
od_cancer = process_od_cancer_sites(df_od)

In [37]:
split_date = "2015-12-31"
target_ae_cols = list(target_dict_mapping.values())

paper_table, paper_table_subset = build_characteristics_table(
    df_id,
    df_od,
    id_cancer,
    od_cancer,
    id_chemo,
    od_chemo,
    target_ae_cols,
    split_date,
    mrn_inference_aero
)

Index(['CISPLATIN', 'FLUOROURACIL', 'GEMCITABINE', 'LEUCOVORIN',
       'CARBOPLATIN'],
      dtype='object', name='Standardized Drug')
Index(['CISPLATIN', 'CARBOPLATIN', 'PACLITAXEL', 'FLUOROURACIL', 'LEUCOVORIN'], dtype='object', name='Standardized Drug')


In [38]:
paper_table_subset

,Training,Temporal OOD,Multi-factor OOD,OOD Aero,OOD Other
"Age, median (IQR)",63.00 (55.00-70.00),65.00 (57.00-71.00),63.17 (52.91-71.69),64.35 (56.48-71.78),61.58 (47.75-71.55)
Female,"1,906 (40.99)",895 (39.64),"1,235 (46.64)",554 (38.00),681 (57.23)
Hb,704 (19.73),411 (21.80),598 (23.74),346 (24.61),252 (22.64)
ANC,"1,224 (34.40)",506 (26.86),427 (16.95),268 (19.06),159 (14.29)
Bili,96 (2.83),45 (2.46),45 (1.88),31 (2.34),14 (1.32)
PLT,246 (6.90),108 (5.73),137 (5.44),80 (5.69),57 (5.12)
ALT,134 (3.90),44 (2.38),82 (3.30),52 (3.74),30 (2.74)
AST,89 (2.58),35 (1.86),55 (2.21),35 (2.52),20 (1.83)
AKI,57 (1.63),30 (1.59),71 (2.83),45 (3.21),26 (2.36)
30d death,84 (1.82),32 (1.43),49 (1.88),35 (2.45),14 (1.19)
